In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

In [5]:
df = pd.read_csv("../data/processed/insurance_clean.csv")

df.head()

,customer_id,age,gender,marital_status,occupation,income_level,education_level,geographic_information,location,behavioral_data,...,previous_claims_history,credit_score,driving_record,life_events,segmentation_group,age_group,credit_score_group,premium_coverage_ratio,estimated_value,risk_category
0,84966,23,Female,Married,Entrepreneur,70541,Associate Degree,Mizoram,37534,policy5,...,3,728,DUI,Job Change,Segment5,18-25,Good,0.007499,5498,Low
1,95568,26,Male,Widowed,Manager,54168,Doctorate,Goa,63304,policy5,...,2,792,Clean,Retirement,Segment5,26-35,Very Good,0.002520,3932,Low
2,10544,29,Female,Single,Entrepreneur,73899,Associate Degree,Rajasthan,53174,policy5,...,1,719,Accident,Childbirth,Segment3,26-35,Good,0.005702,13239,Medium
3,77033,20,Male,Divorced,Entrepreneur,63381,Bachelor's Degree,Sikkim,22803,policy5,...,0,639,DUI,Job Change,Segment3,18-25,Fair,0.005511,17368,High
4,88160,25,Female,Separated,Manager,38794,Bachelor's Degree,West Bengal,92858,policy1,...,3,720,Major Violations,Childbirth,Segment2,18-25,Good,0.003482,1276,Very Low


In [6]:
df.columns

Index(['customer_id', 'age', 'gender', 'marital_status', 'occupation',
       'income_level', 'education_level', 'geographic_information', 'location',
       'behavioral_data', 'purchase_history', 'policy_start_date',
       'policy_renewal_date', 'claim_history',
       'interactions_with_customer_service', 'insurance_products_owned',
       'coverage_amount', 'premium_amount', 'deductible', 'policy_type',
       'customer_preferences', 'preferred_communication_channel',
       'preferred_contact_time', 'preferred_language', 'risk_profile',
       'previous_claims_history', 'credit_score', 'driving_record',
       'life_events', 'segmentation_group', 'age_group', 'credit_score_group',
       'premium_coverage_ratio', 'estimated_value', 'risk_category'],
      dtype='object')

In [7]:
df["high_risk"] = np.where(df["risk_profile"] == 3, 1, 0)

df["high_risk"].value_counts()

high_risk
0    36856
1    16647
Name: count, dtype: int64

In [8]:
features = [
    "age",
    "income_level",
    "claim_history",
    "previous_claims_history",
    "credit_score",
    "coverage_amount",
    "premium_amount",
    "deductible",
    "gender",
    "marital_status",
    "occupation",
    "education_level",
    "policy_type",
    "segmentation_group",
    "behavioral_data",
    "purchase_history"
]

target = "high_risk"

In [9]:
X = df[features]
y = df[target]

In [10]:
numeric_features = [
    "age",
    "income_level",
    "claim_history",
    "previous_claims_history",
    "credit_score",
    "coverage_amount",
    "premium_amount",
    "deductible"
]

categorical_features = [
    "gender",
    "marital_status",
    "occupation",
    "education_level",
    "policy_type",
    "segmentation_group",
    "behavioral_data",
    "purchase_history"
]

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [13]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

In [14]:
logistic_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
y_pred_logistic = logistic_model.predict(X_test)
y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

In [16]:
print("Accuracy:", accuracy_score(y_test, y_pred_logistic))
print("ROC AUC:", roc_auc_score(y_test, y_prob_logistic))
print(classification_report(y_test, y_pred_logistic))

Accuracy: 0.6779740211195215
ROC AUC: 0.5404976404976405
              precision    recall  f1-score   support

           0       0.69      0.97      0.81      7371
           1       0.34      0.04      0.07      3330

    accuracy                           0.68     10701
   macro avg       0.52      0.50      0.44     10701
weighted avg       0.58      0.68      0.58     10701



In [17]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

In [18]:
rf_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [20]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, y_prob_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.6883468834688347
ROC AUC: 0.5492335029372066
              precision    recall  f1-score   support

           0       0.69      1.00      0.82      7371
           1       0.40      0.00      0.01      3330

    accuracy                           0.69     10701
   macro avg       0.54      0.50      0.41     10701
weighted avg       0.60      0.69      0.56     10701



In [21]:
cm = confusion_matrix(y_test, y_pred_rf)

cm

array([[7356,   15],
       [3320,   10]])

In [22]:
cm_df = pd.DataFrame(
    cm,
    index=["Actual Not High Risk", "Actual High Risk"],
    columns=["Predicted Not High Risk", "Predicted High Risk"]
)

cm_df

,Predicted Not High Risk,Predicted High Risk
Actual Not High Risk,7356,15
Actual High Risk,3320,10


In [23]:
rf = rf_model.named_steps["model"]

feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

importance_df.head(20)

,feature,importance
6,num__premium_amount,0.060328
5,num__coverage_amount,0.060036
1,num__income_level,0.060033
7,num__deductible,0.059995
4,num__credit_score,0.059038
0,num__age,0.053277
2,num__claim_history,0.032587
3,num__previous_claims_history,0.027547
37,cat__segmentation_group_Segment5,0.010561
31,cat__policy_type_Group,0.010439


In [24]:
import plotly.express as px

top_features = importance_df.head(15)

fig = px.bar(
    top_features,
    x="importance",
    y="feature",
    orientation="h",
    title="Top 15 Features Influencing High Risk Prediction"
)

fig.show()

## Model Interpretation

The model predicts whether a customer is high risk based on customer demographics, policy characteristics, claims history, credit score, and behavioral variables.
The most important predictors help identify which factors are most strongly associated with high-risk customers. This can support insurance teams in pricing decisions, customer segmentation, and risk monitoring.

In [25]:
import joblib
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

joblib.dump(rf_model, "../models/high_risk_prediction_model.pkl")

['../models/high_risk_prediction_model.pkl']

In [26]:
importance_df.to_csv("../reports/feature_importance.csv", index=False)